# Nordic statistics APIs — hands-on tutorial

**Goal:** Explore Iceland's PxWeb v1 and Sweden's PxWeb v2, fetch samples, and prepare a defensible comparison. This notebook has no saved data output; cells query the live APIs when you run them.

Work in order. API value **codes** are important: they are not always the same as English labels. The `README.md` explains the underlying request formats and statistical caveats.

## Step 1 — Connect

Run Jupyter from the project directory after installing `requirements.txt` and `jupyterlab` (optional). If necessary: `pip install jupyterlab`. No login/API keys required.

In [ ]:
import pandas as pd
from pxweb_client import IcelandPxWeb, SwedenPxWeb, comparable_time_series

pd.set_option("display.max_colwidth", 100)
ice = IcelandPxWeb()
swe = SwedenPxWeb(language="en")

IS_TABLE = "Ibuar/mannfjoldi/1_yfirlit/Yfirlit_mannfjolda/MAN00101.px"
SE_TABLE = "TAB638"

## Step 2 — Discover tables

Iceland uses a directory tree: browse directories until you see a table (type `t`). Sweden v2 lets you search tables by topic; each table has a stable id like `TAB638`.

In [ ]:
display(ice.browse("Ibuar/mannfjoldi/1_yfirlit/Yfirlit_mannfjolda").head(10))
display(swe.search("population", page_size=10)[["id", "label"]].head(10))

## Step 3 — Inspect metadata **before** downloading

Inspect dimension codes, labels and example values. You must select all dimensions explicitly when using this teaching client, so there is no accidental aggregation.

In [ ]:
display(ice.variables(IS_TABLE))
display(swe.variables(SE_TABLE))
# Show every code and label for the Swedish country/region variable.
display(swe.value_codes(SE_TABLE, "Region").head(20))

## Step 4 — Fetch small slices

These sample selections use the first value in every non-time dimension and the most recent five time periods. They are useful to test connectivity, **not** to compare population totals. Always inspect the exact chosen categories.

In [ ]:
is_sel = ice.example_selection(IS_TABLE, last_n=5)
se_sel = swe.example_selection(SE_TABLE, last_n=5)
print("Iceland sample selection:", is_sel)
print("Sweden sample selection:", se_sel)

df_is = ice.fetch(IS_TABLE, is_sel)
df_se = swe.fetch(SE_TABLE, se_sel)
display(df_is)
display(df_se)

## Step 5 — Design a real comparison

Choose one metric. For total population, check category definitions and whether *Total* exists. Confirm the national geography, units, and definitions. **Crucially**, Iceland's January 1 population reference date matches Sweden's December 31 of the **preceding** calendar year. Harmonize those years before merging.

Use `.value_codes(table, variable)` for each dimension, then call `.fetch(table, selected_codes)` with your exact choices. Save the harmonized one-row-per-year series to `data/iceland_matched.csv` (columns `Year,value`) and `data/sweden_matched.csv` (`Tid,value`). These two files must be comparable, not just convenient to fetch.

In [ ]:
# Explore the allowed categories to construct your explicit, matched selection:
for v in ice.variables(IS_TABLE)["code"]:
    print("ICELAND", v)
    display(ice.value_codes(IS_TABLE, v).head(12))
for v in swe.variables(SE_TABLE)["code"]:
    print("SWEDEN", v)
    display(swe.value_codes(SE_TABLE, v).head(12))

## Step 6 — Compare only harmonized data

After creating the matched CSV files as explained above, run the following cell. This refuses duplicate periods, so accidentally downloading age/sex subdivisions cannot silently create a many-to-many join.

In [ ]:
from pathlib import Path

is_file = Path("data/iceland_matched.csv")
se_file = Path("data/sweden_matched.csv")
if is_file.exists() and se_file.exists():
    a = pd.read_csv(is_file, dtype={"Year": str})
    b = pd.read_csv(se_file, dtype={"Tid": str})
    joined = comparable_time_series(a, b, "Year", "Tid", period_to_year=True)
    # Assumes reference dates have ALREADY been aligned when creating these CSVs!
    for country in ["Iceland", "Sweden"]:
        joined[country + "_growth"] = joined[country].pct_change() * 100
    display(joined)
    print("Correlation (growth):", joined[["Iceland_growth", "Sweden_growth"]].corr().iloc[0, 1])
    ax = joined.plot(x="period", y=["Iceland_growth", "Sweden_growth"], marker="o",
                     title="Annual growth in matched indicator", figsize=(9, 5))
    ax.set(xlabel="Reference year", ylabel="Change (%)")
    ax.grid(alpha=0.25)
else:
    print("First create the two harmonized CSV files; see README.md and Step 5.")

## Challenge

1. Change Sweden's `language` to `sv`. Which variable *codes* stay the same, and which labels change?
2. Find an Icelandic mortality or inflation table. What dimensions make direct country comparisons difficult?
3. For a matched 10-year population series, compare Pearson correlation of population levels and of annual growth rates. Why may they differ?

**Interpretation:** Correlation measures association, not causality; changing definitions and shared time trends can produce spurious results.